# Notebook 1 — Edge-case synthesis walkthrough

Build one synthetic edge-case from a **real** photo:

`image → depth → seg → edit → annotate → VLM judge`

**Domain story:** street scenes from [Mapillary Vistas v2](https://www.mapillary.com/dataset/vistas) (CC BY-NC-SA).
Long-tail classes we synthesize: **pothole**, **traffic cone**, **ground animal**.

### Hardware (config only)

| Profile | Edit | Judge |
|---------|------|-------|
| `cpu` | SD 1.5 **inpaint** | Qwen2.5-VL-3B |
| `gpu_l4` | **SDXL inpaint** | Qwen2.5-VL-7B |

Swap with `HARDWARE = "gpu_l4"` below. Model IDs live in `configs/hardware/*.yaml`.

> Seg is used to keep edits on the road. Depth is optional (ControlNet). The **VLM judge only sees RGB + text**.


---
## 0. Setup

```bash
cd implementations/edge_case_image_generation && uv sync
```

Select the `EdgeCase Synthesis` kernel, then run:


In [ ]:
import sys
from pathlib import Path

def _find_project_root() -> Path:
    """Locate edge_case_image_generation root (has src/edgecase_synthesis + configs)."""
    here = Path.cwd().resolve()
    search = [here, *here.parents]
    # Common Cursor/Jupyter cwd: monorepo root — look one level down.
    for base in list(search):
        nested = base / "implementations" / "edge_case_image_generation"
        if nested.is_dir():
            search.append(nested)
    for base in search:
        if (base / "src" / "edgecase_synthesis").is_dir() and (base / "configs").is_dir():
            return base
        if base.name == "notebooks" and (base.parent / "src" / "edgecase_synthesis").is_dir():
            return base.parent
    raise FileNotFoundError(
        "Could not find edge_case_image_generation project root.\n"
        f"cwd={here}\n"
        "Open notebooks from implementations/edge_case_image_generation/notebooks/ "
        "and select the EdgeCase Synthesis kernel (.venv)."
    )

PROJECT_ROOT = _find_project_root()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
print("Project root:", PROJECT_ROOT)
import edgecase_synthesis
print("Package:", edgecase_synthesis.__version__)


---
## 1. Load real images

Default source: **Mapillary Vistas** toy subset (`configs/data/source/mapillary_vistas.yaml`).
Frames + `labels.json` live under `data/samples/` (built by `scripts/extract_mapillary_toy.py` — no full 29 GB download).

Alternatives: `rdd2022`, `local`, `urls`, `nordland_hf`.


In [ ]:
from edgecase_synthesis.config import load_config
from edgecase_synthesis.data import (
    get_data_source_info,
    load_detection_labels,
    load_sample_images,
    prepare_sample_images,
)
from edgecase_synthesis.viz import show_samples

HARDWARE = "cpu"  # or "gpu_l4"
overrides = [f"hardware={HARDWARE}"]
# overrides.append("data/source@data=rdd2022")  # optional alternate

cfg = load_config(start=PROJECT_ROOT, overrides=overrides)
info = get_data_source_info(cfg)
samples_dir = Path(cfg.paths.samples_dir)
samples_dir.mkdir(parents=True, exist_ok=True)

print(f"Hardware: {cfg.hardware.name}")
print(f"Source: {info.label}  ({info.license})")
print(f"Depth: {cfg.conditioning.depth.model_id}")
print(f"Edit: family={cfg.generation.family}  method={cfg.generation.default_anomaly_method}")
print(f"Judge: {cfg.judge.model_id}")

# Mapillary is local: if data/samples/ is empty, prepare_sample_images runs
# scripts/extract_mapillary_toy.py (requires `huggingface-cli login`).
prepare_sample_images(samples_dir, cfg=cfg)
samples = load_sample_images(samples_dir, cfg=cfg)
gt_labels = load_detection_labels(samples_dir, cfg=cfg)
print(f"Loaded {len(samples)} images; {len(gt_labels)} have cached GT boxes")
show_samples(samples, ncol=min(4, len(samples)), figsize=(14, 4));


---
## 2. Depth

Relative depth for ControlNet / placement. Model id from hardware config.


In [ ]:
from edgecase_synthesis.conditioning import DepthEstimator
from edgecase_synthesis.viz import show_depth_result

depth_model = DepthEstimator.from_config(cfg)
print(depth_model.model_id, "on", depth_model.device)
depth_results = {s.name: depth_model.predict(s.image) for s in samples}
show_depth_result(samples[0], depth_results[samples[0].name]);


---
## 3. Segmentation

Generic semantic map (for viz / optional mask intersection). Class ids for "ground" come from YAML if set.


In [ ]:
from edgecase_synthesis.conditioning import Segmenter
from edgecase_synthesis.viz import show_segmentation_result

segmenter = Segmenter.from_config(cfg)
print(segmenter.model_name, "on", segmenter.device)
seg_results = {s.name: segmenter.predict(s.image) for s in samples}
show_segmentation_result(samples[0], seg_results[samples[0].name]);


---
## 4. Structure overview


In [ ]:
from edgecase_synthesis.viz import save_structure_artifacts, show_structure_overview

sample = samples[0]
show_structure_overview(sample, depth_results[sample.name], seg_results[sample.name]);
save_structure_artifacts(sample, depth_results[sample.name], seg_results[sample.name], cfg.paths.outputs_dir);


---
## 5. Anomaly edit

Anomaly prompts + edit masks are YAML under `configs/generation/anomalies/<dataset>/`.  
`method: auto` → hardware default (**inpaint** on CPU and L4; depth ControlNet optional).


In [ ]:
from edgecase_synthesis.config import list_anomalies, load_anomaly, merge_generation_anomaly
from edgecase_synthesis.generation import AnomalyEditor, _resolve_method
from edgecase_synthesis.viz import save_generation_artifact, show_generation_result

editor = AnomalyEditor.from_config(cfg)
dataset = str(cfg.generation.anomaly_dataset)
workshop = list(cfg.generation.workshop_anomalies)
output_dir = Path(cfg.paths.outputs_dir)
print("family=", editor.family, "device=", editor.device)
print("anomalies:", workshop, "| on disk:", list_anomalies(dataset, start=PROJECT_ROOT))

generated_by_anomaly = {}
for anomaly_id in workshop:
    anomaly_cfg = load_anomaly(dataset, anomaly_id, start=PROJECT_ROOT)
    method = _resolve_method(merge_generation_anomaly(cfg.generation, anomaly_cfg).anomaly, merge_generation_anomaly(cfg.generation, anomaly_cfg))
    print("=" * 60, f"\n{anomaly_cfg.get('display_name', anomaly_id)} | method={method}")
    generated = editor.generate_anomaly(
        sample.image,
        depth_results[sample.name],
        seg_results[sample.name],
        cfg.generation,
        anomaly_cfg,
    )
    generated_by_anomaly[anomaly_id] = generated
    show_generation_result(sample, generated)
    print(save_generation_artifact(sample, generated, output_dir))


---
## 6. Auto-annotation

Open-vocabulary detector + SAM (ids from hardware YAML). Seed the edit mask when the detector misses the synthetic defect.


In [ ]:
from edgecase_synthesis.annotation import OpenVocabAnnotator
from edgecase_synthesis.viz import save_annotation_artifact, show_annotation_result

annotator = OpenVocabAnnotator.from_config(cfg)
print(annotator.detector_model, "+", annotator.sam_model)
base_classes = list(cfg.annotation.classes)
annotations_by_anomaly = {}

for anomaly_id, generated in generated_by_anomaly.items():
    anomaly_cfg = load_anomaly(dataset, anomaly_id, start=PROJECT_ROOT)
    anomaly_classes = list(anomaly_cfg.get("annotation_classes", []))
    classes = list(dict.fromkeys([*base_classes, *anomaly_classes]))
    seed_label = next((c for c in anomaly_classes if c not in {"road"}), None)
    annotation = annotator.annotate(
        generated.image,
        classes=classes,
        seed_mask=generated.edit_mask if seed_label else None,
        seed_label=seed_label,
    )
    annotations_by_anomaly[anomaly_id] = annotation
    show_annotation_result(generated.image, annotation, title=f"Annotations — {anomaly_id}")
    print(save_annotation_artifact(f"{sample.name}_{anomaly_id}", annotation, output_dir))


---
## 7. VLM judge

Scores RGB + prompt (+ annotation summary). Unload heavy models first on GPU.


In [ ]:
import gc
import torch
from edgecase_synthesis.judge import VLMJudge, summarize_annotations
from edgecase_synthesis.viz import save_judge_artifact, show_judge_result

for name in ("editor", "annotator", "depth_model", "segmenter"):
    obj = globals().get(name)
    if obj is not None and hasattr(obj, "unload"):
        obj.unload()
    globals()[name] = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

judge = VLMJudge.from_config(cfg)
print(judge.model_id, "threshold=", judge.threshold)
source_hint = str(cfg.judge.get("source_hint", "a real industrial surface photograph"))

judgments = {}
for anomaly_id, generated in generated_by_anomaly.items():
    anomaly_cfg = load_anomaly(dataset, anomaly_id, start=PROJECT_ROOT)
    result = judge.judge(
        generated.image,
        prompt=generated.prompt,
        anomaly_id=anomaly_id,
        anomaly_name=str(anomaly_cfg.get("display_name", anomaly_id)),
        annotations_summary=summarize_annotations(annotations_by_anomaly.get(anomaly_id)),
        source_hint=source_hint,
    )
    judgments[anomaly_id] = result
    show_judge_result(generated.image, result, title=f"Judge — {anomaly_id}")
    print(result.decision, result.overall, save_judge_artifact(f"{sample.name}_{anomaly_id}", result, output_dir))
    print(result.rationale)

accepted = sum(1 for r in judgments.values() if r.decision == "accept")
print(f"Acceptance: {accepted}/{len(judgments)}")


---
## Wrap-up

Single-image loop is complete. Next:

1. **Notebook 2** — batch + retry on `decision=retry`
2. **Notebook 3** — train a tiny detector on Mapillary **real vs real+accepted synthetic**; report rare-class AP (e.g. **pothole** / **traffic_cone**)

Add a new anomaly: drop a YAML under `configs/generation/anomalies/mapillary_vistas/` and append its id to `generation.workshop_anomalies`.
